Practical 1: Sentiment Analysis – Simple ANN &
Transformers

**Task 1.1**

Preprocessing steps in data_loading_code.py provided

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from google.colab import files
import io
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)

    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace(r'[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)

    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if w not in stopwords.words('english')]

        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }

    return data


if __name__ == "__main__":

    # Load data
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    data = pd.read_csv(io.BytesIO(uploaded[filename]), delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index

    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)

    # Split data
    X = data['Sentence'].values.astype('U')
    y = data['Class'].values.astype('int32')

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.1,
        random_state=0,
        shuffle=True
    )

    # TF-IDF vectorization
    word_vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=50000,
        max_df=0.5,
        use_idf=True,
        norm='l2'
    )

    # FIT ONLY ON TRAIN
    X_train_tfidf = word_vectorizer.fit_transform(X_train)
    X_val_tfidf = word_vectorizer.transform(X_val)

    # convert sparse → dense
    X_train_tfidf = X_train_tfidf.todense()
    X_val_tfidf = X_val_tfidf.todense()

    # Convert to tensors
    train_x_tensor = torch.from_numpy(np.array(X_train_tfidf)).float()
    train_y_tensor = torch.from_numpy(np.array(y_train)).long()

    validation_x_tensor = torch.from_numpy(np.array(X_val_tfidf)).float()
    validation_y_tensor = torch.from_numpy(np.array(y_val)).long()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Saving amazon_cells_labelled.txt to amazon_cells_labelled.txt


Data Loading for 25k dataset

In [ ]:
uploaded_25k = files.upload()
filename_25k = list(uploaded_25k.keys())[0]
data_25k = pd.read_csv(io.BytesIO(uploaded_25k[filename_25k]), delimiter='\t', header=None)
data_25k.columns = ['Sentence', 'Class']
data_25k['index'] = data_25k.index

# Run the SAME preprocess_pandas function on this new data
data_25k = preprocess_pandas(data_25k, ['index', 'Class', 'Sentence'])

X_25k = data_25k['Sentence'].values.astype('U')
y_25k = data_25k['Class'].values.astype('int32')

X_train_25k, X_val_25k, y_train_25k, y_val_25k = train_test_split(
    X_25k, y_25k, test_size=0.1, random_state=0
)

# TF-IDF for Big Data (REDUCE features to 10k to prevent crash)
vectorizer_25k = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf_25k = vectorizer_25k.fit_transform(X_train_25k).todense()
X_val_tfidf_25k = vectorizer_25k.transform(X_val_25k).todense()

# Tensors for 25k
train_x_25k = torch.from_numpy(np.array(X_train_tfidf_25k)).float()
train_y_25k = torch.from_numpy(np.array(y_train_25k)).long()

# Tensors for Validation to test accuracy
val_x_25k = torch.from_numpy(np.array(X_val_tfidf_25k)).float()
val_y_25k = torch.from_numpy(np.array(y_val_25k)).long()

Saving amazon_cells_labelled_LARGE_25K.txt to amazon_cells_labelled_LARGE_25K.txt


Model 1: Simple ANN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleANN(nn.Module):
    def __init__(self, input_size):
        super(SimpleANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

**1k dataset**

In [ ]:
model = SimpleANN(train_x_tensor.shape[1])

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(train_x_tensor)
    loss = criterion(outputs, train_y_tensor)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/30, Loss: 0.6938
Epoch 2/30, Loss: 0.6922
Epoch 3/30, Loss: 0.6901
Epoch 4/30, Loss: 0.6877
Epoch 5/30, Loss: 0.6846
Epoch 6/30, Loss: 0.6811
Epoch 7/30, Loss: 0.6768
Epoch 8/30, Loss: 0.6721
Epoch 9/30, Loss: 0.6664
Epoch 10/30, Loss: 0.6598
Epoch 11/30, Loss: 0.6525
Epoch 12/30, Loss: 0.6445
Epoch 13/30, Loss: 0.6355
Epoch 14/30, Loss: 0.6262
Epoch 15/30, Loss: 0.6162
Epoch 16/30, Loss: 0.6048
Epoch 17/30, Loss: 0.5932
Epoch 18/30, Loss: 0.5798
Epoch 19/30, Loss: 0.5667
Epoch 20/30, Loss: 0.5523
Epoch 21/30, Loss: 0.5363
Epoch 22/30, Loss: 0.5198
Epoch 23/30, Loss: 0.5024
Epoch 24/30, Loss: 0.4842
Epoch 25/30, Loss: 0.4665
Epoch 26/30, Loss: 0.4464
Epoch 27/30, Loss: 0.4255
Epoch 28/30, Loss: 0.4057
Epoch 29/30, Loss: 0.3844
Epoch 30/30, Loss: 0.3631


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()

with torch.no_grad():
    outputs = model(validation_x_tensor)
    preds = torch.argmax(outputs, dim=1)

print("\n--- 1k Dataset Results ---")
print("Accuracy:", accuracy_score(validation_y_tensor, preds))
print("Precision:", precision_score(validation_y_tensor, preds))
print("Recall:", recall_score(validation_y_tensor, preds))


--- 1k Dataset Results ---
Accuracy: 0.77
Precision: 0.875
Recall: 0.660377358490566


**25k dataset**

In [ ]:
model_ann_25k = SimpleANN(train_x_25k.shape[1])
optimizer_25k = optim.Adam(model_ann_25k.parameters(), lr=0.001)

epochs_25k = 10
batch_size_25k = 128

train_loader_25k = DataLoader(TensorDataset(train_x_25k, train_y_25k), batch_size=batch_size_25k, shuffle=True)

criterion = nn.CrossEntropyLoss()

model_ann_25k.train()
for epoch in range(epochs_25k):
    total_loss = 0
    for xb, yb in train_loader_25k:
        optimizer_25k.zero_grad()
        outputs = model_ann_25k(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer_25k.step()
        total_loss += loss.item()
    print(f"25k ANN - Epoch {epoch+1}/{epochs_25k}, Loss: {total_loss/len(train_loader_25k):.4f}")

25k ANN - Epoch 1/10, Loss: 0.4004
25k ANN - Epoch 2/10, Loss: 0.1929
25k ANN - Epoch 3/10, Loss: 0.1316
25k ANN - Epoch 4/10, Loss: 0.0856
25k ANN - Epoch 5/10, Loss: 0.0512
25k ANN - Epoch 6/10, Loss: 0.0261
25k ANN - Epoch 7/10, Loss: 0.0125
25k ANN - Epoch 8/10, Loss: 0.0066
25k ANN - Epoch 9/10, Loss: 0.0036
25k ANN - Epoch 10/10, Loss: 0.0027


In [ ]:
model_ann_25k.eval()
with torch.no_grad():
    # Pass the 25k validation set through the 25k model
    outputs_25k = model_ann_25k(val_x_25k)
    preds_25k = torch.argmax(outputs_25k, dim=1)

print("\n--- 25k Dataset Results ---")
print("Accuracy:", accuracy_score(val_y_25k, preds_25k))
print("Precision:", precision_score(val_y_25k, preds_25k))
print("Recall:", recall_score(val_y_25k, preds_25k))


--- 25k Dataset Results ---
Accuracy: 0.8616
Precision: 0.8802395209580839
Recall: 0.8885157824042982


Model 2: LSTM

For LSTM we must use word sequences, TF-IDF is not usable for LSTM

**1k dataset**

In [ ]:
# Tokenize text
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [ ]:
# Convert text to sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

In [ ]:
# padding, make them into same length
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='pre')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='pre')

In [ ]:
# convert to tensors
y_train_tensor = torch.tensor(y_train).long()
y_val_tensor = torch.tensor(y_val).long()

X_train_tensor = torch.tensor(X_train_pad).long()
X_val_tensor = torch.tensor(X_val_pad).long()

In [ ]:
# Create dataloader
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=20000, embed_dim=128, hidden_dim=128, output_dim=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMClassifier().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
model.train()

for epoch in range(5):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        print(f"Epoch {epoch+1}, avg loss: {total_loss / len(train_loader):.4f}")

Epoch 1, avg loss: 0.0234
Epoch 1, avg loss: 0.0472
Epoch 1, avg loss: 0.0706
Epoch 1, avg loss: 0.0942
Epoch 1, avg loss: 0.1184
Epoch 1, avg loss: 0.1420
Epoch 1, avg loss: 0.1658
Epoch 1, avg loss: 0.1896
Epoch 1, avg loss: 0.2130
Epoch 1, avg loss: 0.2357
Epoch 1, avg loss: 0.2582
Epoch 1, avg loss: 0.2820
Epoch 1, avg loss: 0.3054
Epoch 1, avg loss: 0.3278
Epoch 1, avg loss: 0.3502
Epoch 1, avg loss: 0.3721
Epoch 1, avg loss: 0.3959
Epoch 1, avg loss: 0.4187
Epoch 1, avg loss: 0.4415
Epoch 1, avg loss: 0.4631
Epoch 1, avg loss: 0.4863
Epoch 1, avg loss: 0.5092
Epoch 1, avg loss: 0.5323
Epoch 1, avg loss: 0.5546
Epoch 1, avg loss: 0.5761
Epoch 1, avg loss: 0.5986
Epoch 1, avg loss: 0.6214
Epoch 1, avg loss: 0.6430
Epoch 1, avg loss: 0.6687
Epoch 2, avg loss: 0.0198
Epoch 2, avg loss: 0.0412
Epoch 2, avg loss: 0.0599
Epoch 2, avg loss: 0.0796
Epoch 2, avg loss: 0.0986
Epoch 2, avg loss: 0.1171
Epoch 2, avg loss: 0.1354
Epoch 2, avg loss: 0.1527
Epoch 2, avg loss: 0.1718
Epoch 2, avg

In [ ]:
print("sample sequence:", X_train_pad[0])
print("label distribution:", np.bincount(y_train))
print("non-zero ratio:", (X_train_pad > 0).mean())

sample sequence: [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   8 498   2 746  14 747
 748  12   2 125 499   4  51  52   2  91]
label distribution: [453 447]
non-zero ratio: 0.10264444444444444


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()
preds = []
true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)
        predicted = torch.argmax(outputs, dim=1).cpu().numpy()

        preds.extend(predicted)
        true.extend(y_batch.numpy())

print("\n--- 1k Dataset LSTM Results ---")
print("Accuracy:", accuracy_score(true, preds))
print("Precision:", precision_score(true, preds))
print("Recall:", recall_score(true, preds))


--- 1k Dataset LSTM Results ---
Accuracy: 0.8
Precision: 0.7796610169491526
Recall: 0.8679245283018868


**25k dataset**

In [ ]:
tokenizer_25k = Tokenizer(num_words=50000, oov_token="<OOV>")
tokenizer_25k.fit_on_texts(X_train_25k)

X_train_seq_25k = tokenizer_25k.texts_to_sequences(X_train_25k)
X_val_seq_25k = tokenizer_25k.texts_to_sequences(X_val_25k)

X_train_pad_25k = pad_sequences(X_train_seq_25k, maxlen=250, padding='pre')
X_val_pad_25k = pad_sequences(X_val_seq_25k, maxlen=250, padding='pre')

train_x_lstm_25k = torch.tensor(X_train_pad_25k).long()
train_y_lstm_25k = torch.tensor(y_train_25k).long()
val_x_lstm_25k = torch.tensor(X_val_pad_25k).long()
val_y_lstm_25k = torch.tensor(y_val_25k).long()

train_loader_lstm_25k = DataLoader(TensorDataset(train_x_lstm_25k, train_y_lstm_25k), batch_size=128, shuffle=True)
val_loader_lstm_25k = DataLoader(TensorDataset(val_x_lstm_25k, val_y_lstm_25k), batch_size=128)

In [ ]:
model_lstm_25k = LSTMClassifier(vocab_size=50000).to(device)
optimizer_lstm_25k = torch.optim.Adam(model_lstm_25k.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

model_lstm_25k.train()
for epoch in range(5):
    total_loss = 0
    for xb, yb in train_loader_lstm_25k:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_lstm_25k.zero_grad()
        loss = criterion(model_lstm_25k(xb), yb)
        loss.backward()
        optimizer_lstm_25k.step()
        total_loss += loss.item()
    print(f"25k LSTM - Epoch {epoch+1}, Avg Loss: {total_loss/len(train_loader_lstm_25k):.4f}")

model_lstm_25k.eval()
preds_lstm_25k, true_lstm_25k = [], []

with torch.no_grad():
    for xb, yb in val_loader_lstm_25k:
        outputs = model_lstm_25k(xb.to(device))
        preds_lstm_25k.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        true_lstm_25k.extend(yb.numpy())

print("\n--- 25k Dataset LSTM Results ---")
print("Accuracy:", accuracy_score(true_lstm_25k, preds_lstm_25k))
print("Precision:", precision_score(true_lstm_25k, preds_lstm_25k))
print("Recall:", recall_score(true_lstm_25k, preds_lstm_25k))

25k LSTM - Epoch 1, Avg Loss: 0.5162
25k LSTM - Epoch 2, Avg Loss: 0.3574
25k LSTM - Epoch 3, Avg Loss: 0.2950
25k LSTM - Epoch 4, Avg Loss: 0.2155
25k LSTM - Epoch 5, Avg Loss: 0.1544

--- 25k Dataset LSTM Results ---
Accuracy: 0.846
Precision: 0.822429906542056
Recall: 0.9456010745466756


The Simple ANN uses a feed-forward structure with three linear layers to process text as TF-IDF vectors, focusing on word importance rather than sequence. In contrast, the LSTM is a recurrent model that uses learned embeddings to process words in order, allowing it to capture how sentence structure and context influence sentiment.

These models were tested on a 1k dataset for initial prototyping and a 25k dataset for larger-scale training. While the Simple ANN performed efficiently on the smaller set, the 25k dataset provided the volume necessary for the models to generalize better.

**Task 1.2**

Transformer on 1k Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=250):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, nhead=8, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layers = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, 2)

    def forward(self, x):
        x = self.pos_encoder(self.embedding(x) * math.sqrt(128))
        x = self.transformer_encoder(x).mean(dim=1)
        return self.fc(x)

model_1k = TransformerClassifier(vocab_size=20000).to(device)
optimizer_1k = optim.Adam(model_1k.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

train_loader_1k = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
val_loader_1k = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=64)

epochs = 10
for epoch in range(epochs):
    model_1k.train()
    total_loss = 0
    for xb, yb in train_loader_1k:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_1k.zero_grad()
        loss = criterion(model_1k(xb), yb)
        loss.backward()
        optimizer_1k.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader_1k):.4f}")

model_1k.eval()
preds_1k, true_1k = [], []
with torch.no_grad():
    for xb, yb in val_loader_1k:
        output = model_1k(xb.to(device))
        preds_1k.extend(torch.argmax(output, dim=1).cpu().numpy())
        true_1k.extend(yb.numpy())

print("\n--- Final 1k Transformer Results ---")
print(f"Accuracy: {accuracy_score(true_1k, preds_1k):.4f}")
print(f"Precision: {precision_score(true_1k, preds_1k):.4f}")
print(f"Recall: {recall_score(true_1k, preds_1k):.4f}")

Epoch 1/10, Loss: 0.6569
Epoch 2/10, Loss: 0.6141
Epoch 3/10, Loss: 0.5144
Epoch 4/10, Loss: 0.4874
Epoch 5/10, Loss: 0.4464
Epoch 6/10, Loss: 0.4439
Epoch 7/10, Loss: 0.3788
Epoch 8/10, Loss: 0.3283
Epoch 9/10, Loss: 0.3113
Epoch 10/10, Loss: 0.2672

--- Final 1k Transformer Results ---
Accuracy: 0.7900
Precision: 0.8200
Recall: 0.7736


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=250):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, nhead=8, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layers = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, 2)

    def forward(self, x):
        x = self.pos_encoder(self.embedding(x) * math.sqrt(128))
        x = self.transformer_encoder(x).mean(dim=1)
        return self.fc(x)

model_25k = TransformerClassifier().to(device)
optimizer = optim.Adam(model_25k.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

train_loader_25k = DataLoader(TensorDataset(train_x_lstm_25k, train_y_lstm_25k), batch_size=64, shuffle=True)
val_loader_25k = DataLoader(TensorDataset(val_x_lstm_25k, val_y_lstm_25k), batch_size=64)

epochs = 10
for epoch in range(epochs):
    model_25k.train()
    total_loss = 0
    for xb, yb in train_loader_25k:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_25k(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader_25k):.4f}")

model_25k.eval()
preds, true = [], []
with torch.no_grad():
    for xb, yb in val_loader_25k:
        output = model_25k(xb.to(device))
        preds.extend(torch.argmax(output, dim=1).cpu().numpy())
        true.extend(yb.numpy())

print("\n--- Final 25k Transformer Results ---")
print(f"Accuracy: {accuracy_score(true, preds):.4f}")
print(f"Precision: {precision_score(true, preds):.4f}")
print(f"Recall: {recall_score(true, preds):.4f}")

Training on: cuda
Epoch 1/10, Loss: 0.4843
Epoch 2/10, Loss: 0.3804
Epoch 3/10, Loss: 0.3436
Epoch 4/10, Loss: 0.3167
Epoch 5/10, Loss: 0.2983
Epoch 6/10, Loss: 0.2815
Epoch 7/10, Loss: 0.2644
Epoch 8/10, Loss: 0.2467
Epoch 9/10, Loss: 0.2290
Epoch 10/10, Loss: 0.2135

--- Final 25k Transformer Results ---
Accuracy: 0.8600
Precision: 0.8677
Recall: 0.9026


# **Task 1.3**



### **Performance Comparison**

Evaluating the three architectures across the 1k and 25k datasets reveals clear performance shifts based on data volume. On the limited 1k dataset, none of the models achieved a truly balanced performance due to data scarcity.

On the limited 1k dataset, the Simple ANN reached an overall accuracy of $0.77$ but suffered from a poor Recall of $0.66$, missing a significant portion of actual positive reviews. Scaling to the 25k dataset resolved this bottleneck, balancing the model to $0.86$ accuracy and $0.88$ recall.


On the 1k dataset, the LSTM performed marginally better at capturing context, achieving $0.80$ accuracy and $0.86$ recall. On the 25k dataset, it capitalized on its sequential memory to reach an exceptionally high Recall of $0.94$, making it the most thorough at identifying positive sentiment.

On the 1k dataset, the Transformer faced severe data scarcity, achieving $0.79$ accuracy but failing to reliably identify positive sentiment with a Recall of only $0.77$. However, scaling to the 25k dataset allowed its performance to surge to an Accuracy of $0.86$ and a Recall of $0.90$.

### **Scenarios**

- The Simple ANN is preferable for tasks where training speed is critical and the dataset is relatively small, as TF-IDF extracts sufficient keyword signals without needing complex sequence modeling.
- The LSTM is ideal for tasks where word order and sentence structure dictate sentiment, such as detecting double negatives.
- The Transformer is the superior choice for extracting highly complex semantic nuances across long documents, but only when an expansive dataset is available to satisfy its parameter requirements.


### **Complexity, Accuracy, and Efficiency Differences**

- Simple ANN: Represents the lowest architectural complexity, relying on straightforward linear layers and TF-IDF vectorization. It trained the fastest (highest computational efficiency) while maintaining surprisingly robust accuracy across both dataset sizes ($0.77 \rightarrow 0.86$).

- LSTM: Exhibits moderate complexity by processing tokens sequentially. This recurrent nature inherently slows down training efficiency but allows the model to naturally capture context. It outperformed the others in terms of identifying positive instances (Recall) on the larger dataset ($0.94$) because the sequential memory effectively maps the flow of product reviews.

- Transformer: The most complex architecture, utilizing multi-head self-attention and positional encoding. It was the least efficient to train. On the 1k dataset, this complexity was detrimental; the model overfitted and failed to generalize well. However, when provided with 25k samples, the Transformer utilized its vast parameter space to find a deeper global minimum, resulting in a massive leap in its evaluation metrics.

### **Insights on Data Amount, Embeddings, and Architectural Choices**

- Data Amount: The experiments concretely demonstrate the concept of "data hunger." The Transformer requires a massive volume of data to learn how to assign attention weights effectively. The 1k dataset was insufficient, leading to underperformance compared to simpler models. The 25k dataset proved that architectural complexity must scale proportionally with data availability for optimal generalization.

- Embeddings: The ANN relied on TF-IDF, treating reviews as a "bag-of-words" without semantic relationships, which hits a performance ceiling quickly. Both the LSTM and Transformer used learned embeddings initialized from scratch. On the smaller 1k dataset, these embeddings lacked the context to form meaningful semantic clusters. Integrating pre-trained embeddings (like GloVe or BERT) would likely stabilize and boost the 1k runs significantly.

- Architectural Choices: For the Transformer, injecting Positional Encoding was a mandatory architectural choice; without it, the permutation-invariant self-attention mechanism would lose all context of sentence structure. For the LSTM, utilizing pre-padding rather than post-padding was a critical choice that prevented the network from suffering mode collapse (predicting a single class) by ensuring the most relevant, recent information fed directly into the final hidden states.